# Chapter 3 — Paradigms of Machine Learning I: Supervised & Unsupervised Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/winnertechserv/aimlgybs/blob/main/part1_foundations/ch03_paradigms_of_machine_learning_i/ch03_notebook.ipynb)

Companion notebook for **Chapter 3** of *AI-Machine Learning: Get Your Base Strong*. Read the chapter first, then run each cell and compare your output with the output printed in the book (shown below each cell).


In [ ]:
# Optional: install the exact library versions used for the outputs in the book.
# Google Colab already has these libraries; newer versions may change a few digits.
# !pip install -q numpy==2.5.3 pandas==3.0.5 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2 pillow==12.3.0

## Key takeaways

- Machine learning learns patterns from examples. Success is measured by generalization to unseen data.
- Supervised learning uses labeled data to learn y = f(x).
- Classification predicts categories; regression predicts continuous numbers.
- Every supervised model is built in a training phase and judged in a testing phase.
- Unsupervised learning finds structure in unlabeled data: clusters of similar items, or rules about what goes together.
- Association rules are judged by support, confidence and lift. Lift above 1 means a real link.

## Example 1: Supervised Learning — Classification and Regression

Three short supervised tasks, each learned from labeled data. First, a classifier learns to tell children from adults using 20 example people, then labels two new people. Second, a K-NN classifier learns the three Iris species and is tested on flowers it never saw: the test accuracy is what measures generalization. Third, a regression line predicts salary from experience. The answer, salary = 3.04 x years + 32.52, is the same line you will derive by hand in Chapter 7.

In [1]:
# Chapter 3 - Example 1: Supervised learning - classification AND regression
# Both learn a function y = f(x) from LABELED examples, then predict for new x.
import numpy as np
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

# --- 1. Classification: child (0) or adult (1) from height (cm) and weight (kg)
X = np.array([[98, 16.4], [104, 15.9], [110, 19.8], [115, 19.1], [119, 23.0],
              [123, 22.4], [127, 27.1], [131, 26.3], [134, 31.0], [101, 17.5],
              [152, 52.0], [158, 50.5], [161, 63.2], [165, 57.8], [168, 71.0],
              [172, 64.5], [175, 80.3], [179, 72.9], [183, 88.0], [188, 81.4]])
y = np.r_[np.zeros(10, int), np.ones(10, int)]
clf = LogisticRegression(max_iter=1000).fit(X, y)
for person in ([150, 47.0], [110, 20.0]):
    p = clf.predict_proba([person])[0, 1]
    print(f"Height {person[0]} cm, weight {person[1]} kg -> "
          f"{'Adult' if p >= 0.5 else 'Child'} (P(adult) = {p:.2f})")

# --- 2. Multi-class classification: the Iris flowers, with a proper train/test split
iris = load_iris()
Xtr, Xte, ytr, yte = train_test_split(iris.data, iris.target, test_size=0.3,
                                      stratify=iris.target, random_state=1)
knn = KNeighborsClassifier(n_neighbors=5).fit(Xtr, ytr)
print(f"\nIris: trained on {len(ytr)} flowers, tested on {len(yte)} unseen flowers")
print(f"Train accuracy {knn.score(Xtr, ytr):.3f} | Test accuracy {knn.score(Xte, yte):.3f}"
      "  <- the test score measures GENERALIZATION")

# --- 3. Regression: predict a NUMBER (salary in $1000s) from years of experience
years = np.array([1, 3, 4, 6, 7, 9, 11, 13, 16, 20]).reshape(-1, 1)
salary = np.array([35, 40, 47, 50, 56, 58, 67, 71, 82, 93])
reg = LinearRegression().fit(years, salary)
print(f"\nSalary line: salary = {reg.coef_[0]:.2f} x years + {reg.intercept_:.2f}")
print(f"Predicted salary for 10 years: {reg.predict([[10]])[0]:.2f} thousand")

Height 150 cm, weight 47.0 kg -> Adult (P(adult) = 0.96)
Height 110 cm, weight 20.0 kg -> Child (P(adult) = 0.00)

Iris: trained on 105 flowers, tested on 45 unseen flowers
Train accuracy 0.981 | Test accuracy 0.978  <- the test score measures GENERALIZATION

Salary line: salary = 3.04 x years + 32.52
Predicted salary for 10 years: 62.94 thousand


## Example 2: Unsupervised Learning — Clustering People

Here K-means gets the same 20 people but no labels at all. The table shows compactness (inertia) for 1 to 5 clusters. The big drop from 1 to 2 clusters is the 'elbow', so two groups is a natural choice. The two cluster centers come out at exactly the average child (116.2 cm, 21.85 kg) and the average adult (170.1 cm, 68.16 kg). The model found children and adults without ever being told they exist.

In [2]:
# Chapter 3 - Example 2: Unsupervised learning - clustering WITHOUT labels
# K-means groups people by height and weight. We never show it who is a child or an adult.
import numpy as np
from sklearn.cluster import KMeans

X = np.array([[98, 16.4], [104, 15.9], [110, 19.8], [115, 19.1], [119, 23.0],
              [123, 22.4], [127, 27.1], [131, 26.3], [134, 31.0], [101, 17.5],
              [152, 52.0], [158, 50.5], [161, 63.2], [165, 57.8], [168, 71.0],
              [172, 64.5], [175, 80.3], [179, 72.9], [183, 88.0], [188, 81.4]])
hidden_truth = np.r_[np.zeros(10, int), np.ones(10, int)]   # only used to CHECK at the end

print(" k   compactness (inertia)")
for k in range(1, 6):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    print(f" {k}   {km.inertia_:10.1f}")
print("Compactness drops sharply from k=1 to k=2, then flattens (the 'elbow').")

km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)
print("\nCluster centers (height, weight):")
for c in km.cluster_centers_:
    print(f"  ({c[0]:.1f} cm, {c[1]:.2f} kg)")

# Cluster numbers are arbitrary, so match them to the hidden labels before comparing
labels = km.labels_
if np.mean(labels == hidden_truth) < 0.5:
    labels = 1 - labels
print(f"\nAgreement with the hidden child/adult labels: {np.mean(labels == hidden_truth):.0%}")
print("New person (130 cm, 30 kg) joins cluster with center",
      np.round(km.cluster_centers_[km.predict([[130, 30]])[0]], 1))

 k   compactness (inertia)
 1      29581.3
 2       4332.2
 3       2463.4
 4       1170.9
 5        773.5
Compactness drops sharply from k=1 to k=2, then flattens (the 'elbow').

Cluster centers (height, weight):
  (170.1 cm, 68.16 kg)
  (116.2 cm, 21.85 kg)

Agreement with the hidden child/adult labels: 100%
New person (130 cm, 30 kg) joins cluster with center [116.2  21.8]


## Example 3: Unsupervised Learning — Association Rules

Ten shopping baskets, no labels. The program counts how often items appear together and turns each pair into rules with support, confidence and lift. 'banana → cereal' has confidence 1.00 and lift 2.50: everyone who bought bananas also bought cereal. 'milk → bread' has a lift below 1, so those two items are not really linked, even though both are popular.

In [3]:
# Chapter 3 - Example 3: Unsupervised learning - association rules (market-basket analysis)
# Find products that are bought together, using support, confidence and lift.
from itertools import combinations

baskets = [
    {"bread", "milk"}, {"bread", "butter", "milk"}, {"bread", "butter"},
    {"milk", "cereal"}, {"bread", "butter", "jam"}, {"milk", "cereal", "banana"},
    {"bread", "butter", "milk"}, {"cereal", "banana"}, {"bread", "jam"},
    {"milk", "cereal", "bread"},
]
N = len(baskets)
items = sorted(set().union(*baskets))

def support(itemset):
    return sum(itemset <= b for b in baskets) / N   # share of baskets containing ALL items

print("Item supports:", {i: support({i}) for i in items})

rules = []
for a, b in combinations(items, 2):
    s_ab = support({a, b})
    if s_ab < 0.2:                       # ignore rare pairs (minimum support = 20%)
        continue
    for x, z in ((a, b), (b, a)):
        conf = s_ab / support({x})       # P(z in basket | x in basket)
        lift = conf / support({z})       # >1 means 'more often together than by chance'
        rules.append((lift, conf, s_ab, x, z))

print(f"\nRules with support >= 20% (sorted by lift):")
print("  rule                    support  confidence  lift")
for lift, conf, s, x, z in sorted(rules, reverse=True):
    print(f"  {x:>7s} -> {z:<12s}   {s:.2f}      {conf:.2f}     {lift:.2f}")

Item supports: {'banana': 0.2, 'bread': 0.7, 'butter': 0.4, 'cereal': 0.4, 'jam': 0.2, 'milk': 0.6}

Rules with support >= 20% (sorted by lift):
  rule                    support  confidence  lift
   banana -> cereal         0.20      1.00     2.50
   cereal -> banana         0.20      0.50     2.50
   butter -> bread          0.40      1.00     1.43
      jam -> bread          0.20      1.00     1.43
    bread -> butter         0.40      0.57     1.43
    bread -> jam            0.20      0.29     1.43
     milk -> cereal         0.30      0.50     1.25
   cereal -> milk           0.30      0.75     1.25
     milk -> bread          0.40      0.67     0.95
    bread -> milk           0.40      0.57     0.95
   butter -> milk           0.20      0.50     0.83
     milk -> butter         0.20      0.33     0.83
